# テキスト分析｜研究メモの実践：作業日誌の「困りごと」を数え、原文で確かめる
**文字を数える。意味は、確かめる。** ― 小規模農業法人へのAIエージェント導入に向けた予備分析

テキスト分析 第1回（E06〜E09）の手順を、そのまま**農場の作業日誌**に当てはめたノートです。骨格はリポジトリ共通の3段構えです。

| 段 | このノートでやること | 研究メモとの対応 |
|---|---|---|
| 0 準備 | 日誌を読み込み、**分析の前に匿名化**して、残りが0件か確かめる | 注意点③ |
| ① 計算する | 「遅れ」「故障」「人手」の**回数と件数を分けて**数える（分母つき） | 手順・注意点① |
| ① 計算する | KWIC で前後を読み、**否定の形**を規則で判定 → 人の判定と**一致率**で比べる | 注意点② |
| ② 可視化する | 確かめた困りごとを**作業×月**のヒートマップにする | 問い（どの作業・時期に） |
| ③ 判断する | 自分の条件で、AIエージェントを**最初に入れる作業**を決める（答えは人によって違う） | 研究の目的 |

> ⚠️ 日誌36件・氏名・取引先名は**すべて教材用の架空データ**です。実在の農業法人・従業員を表しません。
> 実データで使うときは、**このノートを手元（またはColab）で動かし、日誌の原文を外部の生成AIに貼らない**でください。
> 各段の最後に **自己チェック（`assert`）** があり、記録が出力と食い違うと止まります。データを差し替えたら記録欄も書き直します。

**使い方**：「ランタイム」→「すべてのセルを実行」。追加インストールは不要です（pandas・plotly は Colab に入っています）。


## 0｜データを読み込み、分析の前に匿名化する

In [1]:
import re
import pandas as pd

# 架空の作業日誌（ID, 日付, 記入者, 作業, 本文）。実データはCSVを読み込んで差し替える：
# raw = pd.read_csv('work_diary.csv')   # 列：ID,日付,記入者,作業,本文
DIARY = [
    ('D01', '2026-07-03', '記入者A', '除草', '雨で除草が遅れた。山田さんと2人で午後に回した。'),
    ('D02', '2026-07-05', '記入者B', '出荷', 'みどり青果への出荷は予定どおり。遅れはない。'),
    ('D03', '2026-07-08', '記入者A', '機械整備', '草刈機が故障。刃の交換で直るか佐藤さんに相談。'),
    ('D04', '2026-07-10', '記入者C', '収穫', '暑さで作業が遅れた。人手が足りない。'),
    ('D05', '2026-07-14', '記入者B', '除草', '除草は順調。人手は足りた。'),
    ('D06', '2026-07-17', '記入者A', '機械整備', '管理機の点検。故障はなかった。'),
    ('D07', '2026-07-21', '記入者C', '収穫', '収穫量が多く、手が足りない。'),
    ('D08', '2026-07-24', '記入者B', '出荷', '箱詰めに時間がかかり、出荷が遅れた。'),
    ('D09', '2026-07-28', '記入者A', '除草', '特に問題なし。'),
    ('D10', '2026-07-31', '記入者C', '機械整備', '草刈機がまた故障。先週の故障と同じ所。'),
    ('D11', '2026-08-04', '記入者A', '収穫', '早朝に収穫。暑さ対策で休憩を増やした。'),
    ('D12', '2026-08-07', '記入者B', '出荷', 'ひかり製パンへの納品が1時間遅れた。'),
    ('D13', '2026-08-11', '記入者C', '除草', '人手が足りないので、除草は来週に延期。'),
    ('D14', '2026-08-14', '記入者A', '収穫', '台風の前に急いで収穫。人手を増やしてほしい。'),
    ('D15', '2026-08-18', '記入者B', '機械整備', 'かん水ポンプが動かなくなった。'),
    ('D16', '2026-08-21', '記入者C', '出荷', '出荷は予定どおり。'),
    ('D17', '2026-08-25', '記入者A', '除草', '遅れを取り戻すため、山田さんに応援を頼んだ。'),
    ('D18', '2026-08-28', '記入者B', '収穫', '収穫の遅れはない。'),
    ('D19', '2026-09-01', '記入者C', '機械整備', 'トラクターの故障で耕うんが半日止まった。'),
    ('D20', '2026-09-04', '記入者A', '播種', '播種は順調。'),
    ('D21', '2026-09-08', '記入者B', '播種', '雨で播種が遅れた。'),
    ('D22', '2026-09-11', '記入者C', '出荷', '佐藤さんが休みで人手が足りず、出荷が遅れた。'),
    ('D23', '2026-09-15', '記入者A', '育苗', '育苗ハウスの換気扇が壊れた。'),
    ('D24', '2026-09-18', '記入者B', '播種', '播種機の故障かと思ったが、種詰まりだった。'),
    ('D25', '2026-09-22', '記入者C', '除草', '除草は人手が足りたので早く終わった。'),
    ('D26', '2026-09-25', '記入者A', '育苗', '苗の生育に遅れが見られる。'),
    ('D27', '2026-09-29', '記入者B', '出荷', 'みどり青果から追加注文。人手が足りないかもしれない。'),
    ('D28', '2026-10-02', '記入者C', '播種', '小麦の播種準備。特に問題なし。'),
    ('D29', '2026-10-06', '記入者A', '機械整備', '故障していた草刈機が修理から戻った。'),
    ('D30', '2026-10-09', '記入者B', '播種', '播種が2日遅れ。'),
    ('D31', '2026-10-13', '記入者C', '収穫', 'さつまいもの収穫。人手は十分。'),
    ('D32', '2026-10-16', '記入者A', '育苗', '遅延なし。'),
    ('D33', '2026-10-20', '記入者B', '播種', '小麦の播種。遅れなし。'),
    ('D34', '2026-10-23', '記入者C', '出荷', '出荷の遅れでひかり製パンに連絡した。'),
    ('D35', '2026-10-27', '記入者A', '機械整備', '故障なし。'),
    ('D36', '2026-10-30', '記入者B', '除草', '除草が遅れている。人手が足りない。'),
]
raw = pd.DataFrame(DIARY, columns=['ID', '日付', '記入者', '作業', '本文'])
raw['日付'] = pd.to_datetime(raw['日付'])
raw['月'] = raw['日付'].dt.strftime('%Y-%m')
print(len(raw), '件 ／', raw['月'].min(), '〜', raw['月'].max())
raw.head(3)

36 件 ／ 2026-07 〜 2026-10


,ID,日付,記入者,作業,本文,月
0,D01,2026-07-03,記入者A,除草,雨で除草が遅れた。山田さんと2人で午後に回した。,2026-07
1,D02,2026-07-05,記入者B,出荷,みどり青果への出荷は予定どおり。遅れはない。,2026-07
2,D03,2026-07-08,記入者A,機械整備,草刈機が故障。刃の交換で直るか佐藤さんに相談。,2026-07


**匿名化の辞書**：氏名と取引先名を、分析の**前に**置き換えます。記入者の列も最初から「記入者A」のような仮名にしておきます。
辞書に載っていない名前は残るので、置き換えたあとに**残りが0件か**を必ず確かめます（0件でなければ先に進まない）。


In [2]:
NAMES = ['山田', '佐藤']          # 氏名（敬称の前の部分）
PARTNERS = ['みどり青果', 'ひかり製パン']   # 取引先名

def mask(text):
    for w in sorted(PARTNERS, key=len, reverse=True):
        text = text.replace(w, '［取引先］')
    for w in sorted(NAMES, key=len, reverse=True):
        text = text.replace(w, '［人名］')
    return text

df = raw.copy()
df['本文'] = df['本文'].map(mask)

left = [w for w in NAMES + PARTNERS if df['本文'].str.contains(w).any()]
print('匿名化のあとに残った語：', left if left else 'なし')
assert not left, '匿名化が終わっていません。辞書に追加してから進んでください'
print('置き換えた日誌：', (raw['本文'] != df['本文']).sum(), '件')
del raw   # 以降は匿名化済みの df だけを使う

匿名化のあとに残った語： なし
置き換えた日誌： 8 件


## ①｜現象を計算する：3語の回数と件数（分母つき）
記録は第1回と同じ **4点セット**（検索規則／単位と分母／該当ID／原文の根拠）。`search()` は表示するだけでなく**結果を返す**ので、記録表をコードで作れます。


In [3]:
KEYWORDS = ['遅れ', '故障', '人手']   # 研究メモで決めた3語

def search(word, data=None, show=True):
    data = df if data is None else data
    n = data['本文'].str.count(re.escape(word))
    hit = data[n > 0].assign(回数=n[n > 0])
    r = {'語': word, '回数': int(n.sum()), '件数': len(hit), '分母': len(data), 'ID': hit['ID'].tolist()}
    if show:
        print(f"検索規則：部分文字列「{word}」の一致 ／ 出現 {r['回数']} 回 ／ 該当 {r['件数']} 件（分母 {r['分母']} 件）")
        for _, h in hit.iterrows():
            print(f"  {h['ID']}（{h['回数']}回）[{h['作業']}] {h['本文']}")
    return r

results = {kw: search(kw) for kw in KEYWORDS}

検索規則：部分文字列「遅れ」の一致 ／ 出現 14 回 ／ 該当 14 件（分母 36 件）
  D01（1回）[除草] 雨で除草が遅れた。［人名］さんと2人で午後に回した。
  D02（1回）[出荷] ［取引先］への出荷は予定どおり。遅れはない。
  D04（1回）[収穫] 暑さで作業が遅れた。人手が足りない。
  D08（1回）[出荷] 箱詰めに時間がかかり、出荷が遅れた。
  D12（1回）[出荷] ［取引先］への納品が1時間遅れた。
  D17（1回）[除草] 遅れを取り戻すため、［人名］さんに応援を頼んだ。
  D18（1回）[収穫] 収穫の遅れはない。
  D21（1回）[播種] 雨で播種が遅れた。
  D22（1回）[出荷] ［人名］さんが休みで人手が足りず、出荷が遅れた。
  D26（1回）[育苗] 苗の生育に遅れが見られる。
  D30（1回）[播種] 播種が2日遅れ。
  D33（1回）[播種] 小麦の播種。遅れなし。
  D34（1回）[出荷] 出荷の遅れで［取引先］に連絡した。
  D36（1回）[除草] 除草が遅れている。人手が足りない。
検索規則：部分文字列「故障」の一致 ／ 出現 8 回 ／ 該当 7 件（分母 36 件）
  D03（1回）[機械整備] 草刈機が故障。刃の交換で直るか［人名］さんに相談。
  D06（1回）[機械整備] 管理機の点検。故障はなかった。
  D10（2回）[機械整備] 草刈機がまた故障。先週の故障と同じ所。
  D19（1回）[機械整備] トラクターの故障で耕うんが半日止まった。
  D24（1回）[播種] 播種機の故障かと思ったが、種詰まりだった。
  D29（1回）[機械整備] 故障していた草刈機が修理から戻った。
  D35（1回）[機械整備] 故障なし。
検索規則：部分文字列「人手」の一致 ／ 出現 9 回 ／ 該当 9 件（分母 36 件）
  D04（1回）[収穫] 暑さで作業が遅れた。人手が足りない。
  D05（1回）[除草] 除草は順調。人手は足りた。
  D13（1回）[除草] 人手が足りないので、除草は来週に延期。
  D14（1回）[収穫] 台風の前に急いで収穫。人手を増やしてほしい。
  D22（1回）[出荷] ［人名］さんが休みで人手が足りず、出荷が遅れた。
  D25（1回）

In [4]:
summary = pd.DataFrame(results.values())[['語', '回数', '件数', '分母']]
summary['件数の割合'] = (summary['件数'] / summary['分母']).map('{:.0%}'.format)
summary

,語,回数,件数,分母,件数の割合
0,遅れ,14,14,36,39%
1,故障,8,7,36,19%
2,人手,9,9,36,25%


In [5]:
# 自己チェック：下の記録と出力が一致しているか
assert [(r['回数'], r['件数']) for r in results.values()] == [(14, 14), (8, 7), (9, 9)]
assert results['故障']['回数'] != results['故障']['件数']   # D10 に「故障」が2回
print('①の記録は出力と一致しています。')

①の記録は出力と一致しています。


**記録（①）**：分母は日誌36件（1日1件）。「遅れ」14回・14件、「故障」**8回・7件**、「人手」9回・9件。
「故障」で回数と件数がずれるのは、D10「草刈機が**また故障**。先週の**故障**と同じ所」に2回あるため。
→ この段階で言えるのは「その文字列を含む日誌が何件あるか」だけ。**困りごとの件数ではない**（次で確かめる）。


### KWIC：語の前後を並べて読む
「遅れ」「故障」の**直後**、「人手」の**直後**に何が続くかで意味が反転します。

In [6]:
def kwic(word, width=8):
    for _, r in df.iterrows():
        t, s = r['本文'], r['本文'].find(word)
        while s != -1:
            print(f"{r['ID']}  {t[max(0, s - width):s]:>{width}}【{word}】{t[s + len(word):s + len(word) + width]}")
            s = t.find(word, s + 1)

for kw in KEYWORDS:
    print(f'―― {kw} ――')
    kwic(kw)

―― 遅れ ――
D01     雨で除草が【遅れ】た。［人名］さん
D02  荷は予定どおり。【遅れ】はない。
D04    暑さで作業が【遅れ】た。人手が足りな
D08  がかかり、出荷が【遅れ】た。
D12  への納品が1時間【遅れ】た。
D17          【遅れ】を取り戻すため、
D18       収穫の【遅れ】はない。
D21     雨で播種が【遅れ】た。
D22  が足りず、出荷が【遅れ】た。
D26     苗の生育に【遅れ】が見られる。
D30     播種が2日【遅れ】。
D33    小麦の播種。【遅れ】なし。
D34       出荷の【遅れ】で［取引先］に連
D36       除草が【遅れ】ている。人手が足
―― 故障 ――
D03      草刈機が【故障】。刃の交換で直る
D06   管理機の点検。【故障】はなかった。
D10    草刈機がまた【故障】。先週の故障と同
D10  また故障。先週の【故障】と同じ所。
D19    トラクターの【故障】で耕うんが半日止
D24      播種機の【故障】かと思ったが、種
D29          【故障】していた草刈機が
D35          【故障】なし。
―― 人手 ――
D04  で作業が遅れた。【人手】が足りない。
D05    除草は順調。【人手】は足りた。
D13          【人手】が足りないので、
D14  前に急いで収穫。【人手】を増やしてほしい
D22  名］さんが休みで【人手】が足りず、出荷が
D25       除草は【人手】が足りたので早く
D27  ］から追加注文。【人手】が足りないかもし
D31  つまいもの収穫。【人手】は十分。
D36  草が遅れている。【人手】が足りない。


### 判定基準（読む前に決めておく）
| 判定 | 基準 |
|---|---|
| 困りごと | その語が、作業・出荷・機械・人員の**支障が起きた（起きている）**ことを述べている |
| 否定 | 「遅れはない」「故障なし」「人手は足りた」のように、**支障がなかった**ことを述べている |
| 保留 | 支障の**対象が作業ではない**（例：作物の生育）、**解消の報告**、**予測・推量**など、本文だけでは決められない |


**人の判定**（`HUMAN`）：原文を1件ずつ読んで書いたもの。キーは（日誌ID, 語）、値は（判定, 対象, 根拠となる表現）です。

In [7]:
# 人が原文を読んで決めた判定（日誌×語ごと）
HUMAN = {
    ('D01', '遅れ'): ('困りごと', '作業（除草）', '雨で遅れた'),
    ('D02', '遅れ'): ('否定', '出荷', '遅れはない'),
    ('D04', '遅れ'): ('困りごと', '作業（収穫）', '暑さで遅れた'),
    ('D08', '遅れ'): ('困りごと', '出荷', '箱詰めで遅れた'),
    ('D12', '遅れ'): ('困りごと', '出荷（納品）', '1時間遅れた'),
    ('D17', '遅れ'): ('困りごと', '作業（除草）', '遅れを取り戻すため応援を依頼'),
    ('D18', '遅れ'): ('否定', '作業（収穫）', '遅れはない'),
    ('D21', '遅れ'): ('困りごと', '作業（播種）', '雨で遅れた'),
    ('D22', '遅れ'): ('困りごと', '出荷', '人手不足が原因で遅れた'),
    ('D26', '遅れ'): ('保留', '作物（苗の生育）', '作業の遅れではなく生育の遅れ'),
    ('D30', '遅れ'): ('困りごと', '作業（播種）', '2日遅れ'),
    ('D33', '遅れ'): ('否定', '作業（播種）', '遅れなし'),
    ('D34', '遅れ'): ('困りごと', '出荷', '遅れで取引先に連絡'),
    ('D36', '遅れ'): ('困りごと', '作業（除草）', '遅れている'),
    ('D03', '故障'): ('困りごと', '機械（草刈機）', '故障、刃の交換を相談'),
    ('D06', '故障'): ('否定', '機械（管理機）', '故障はなかった'),
    ('D10', '故障'): ('困りごと', '機械（草刈機）', 'また故障、同じ所'),
    ('D19', '故障'): ('困りごと', '機械（トラクター）', '半日止まった'),
    ('D24', '故障'): ('否定', '機械（播種機）', '故障かと思ったが種詰まり'),
    ('D29', '故障'): ('保留', '機械（草刈機）', '修理から戻った＝解消の報告'),
    ('D35', '故障'): ('否定', '機械', '故障なし'),
    ('D04', '人手'): ('困りごと', '人員', '人手が足りない'),
    ('D05', '人手'): ('否定', '人員', '人手は足りた'),
    ('D13', '人手'): ('困りごと', '人員', '足りないので延期'),
    ('D14', '人手'): ('困りごと', '人員', '増やしてほしい（要望）'),
    ('D22', '人手'): ('困りごと', '人員', '休みで足りず'),
    ('D25', '人手'): ('否定', '人員', '人手が足りた'),
    ('D27', '人手'): ('保留', '人員', '足りないかもしれない（予測）'),
    ('D31', '人手'): ('否定', '人員', '人手は十分'),
    ('D36', '人手'): ('困りごと', '人員', '人手が足りない'),
}
print(len(HUMAN), '組（日誌×語）')
print(pd.Series([v[0] for v in HUMAN.values()]).value_counts().to_dict())

30 組（日誌×語）
{'困りごと': 18, '否定': 9, '保留': 3}


### 機械の判定と比べる
- **素朴な規則**：本文のどこかに「ない・なし・なかった」があれば否定とみなす。
- **規則2**：語の**直後**の形で判定し、しかも**語ごとに別の規則**にする。「遅れ**はない**」は困りごとなしだが、「人手**がない**」「人手が足り**ない**」は困りごと**あり**だからです（第1回の「難しい**わけではない**」と同じ構造）。


In [8]:
NEG = {   # ← 打ち消し・充足の言い方（ここを変えて試せる）
    '遅れ': r'(遅れ|遅延)(は|が|も)?(ない|なし|なかった|無し)',
    '故障': r'故障(は|が|も)?(ない|なし|なかった|無し)',
    '人手': r'人手(は|が|も)?(足りた|足りている|十分)',
}

def naive(text):
    return '否定' if re.search(r'ない|なし|なかった', text) else '困りごと'

def rule2(text, kw):
    return '否定' if re.search(NEG[kw], text) else '困りごと'

rows = []
for (id_, kw), (label, target, reason) in HUMAN.items():
    t = df.loc[df['ID'] == id_, '本文'].item()
    rows.append({'ID': id_, '語': kw, '本文': t, '人': label, '対象': target,
                 '素朴': naive(t), '規則2': rule2(t, kw)})
judge = pd.DataFrame(rows)
for col in ['素朴', '規則2']:
    ok = (judge[col] == judge['人']).sum()
    print(f'{col:<4} 人との一致：{ok} / {len(judge)} 組（{ok / len(judge):.0%}）')

素朴   人との一致：18 / 30 組（60%）
規則2  人との一致：26 / 30 組（87%）


In [9]:
print('素朴な規則が外したもの（「ない」があるのに困りごと）：')
print(judge[(judge['素朴'] != judge['人']) & (judge['人'] == '困りごと')][['ID', '語', '本文']].to_string(index=False))
print()
print('規則2でも人と食い違ったもの：')
print(judge[judge['規則2'] != judge['人']][['ID', '語', '本文', '規則2', '人', '対象']].to_string(index=False))

素朴な規則が外したもの（「ない」があるのに困りごと）：
 ID  語                  本文
D04 遅れ  暑さで作業が遅れた。人手が足りない。
D36 遅れ   除草が遅れている。人手が足りない。
D04 人手  暑さで作業が遅れた。人手が足りない。
D13 人手 人手が足りないので、除草は来週に延期。
D36 人手   除草が遅れている。人手が足りない。

規則2でも人と食い違ったもの：
 ID  語                         本文  規則2  人       対象
D26 遅れ              苗の生育に遅れが見られる。 困りごと 保留 作物（苗の生育）
D24 故障      播種機の故障かと思ったが、種詰まりだった。 困りごと 否定  機械（播種機）
D29 故障         故障していた草刈機が修理から戻った。 困りごと 保留  機械（草刈機）
D27 人手 ［取引先］から追加注文。人手が足りないかもしれない。 困りごと 保留       人員


In [10]:
# 自己チェック
assert len(judge) == 30
assert (judge['素朴'] == judge['人']).sum() == 18
assert (judge['規則2'] == judge['人']).sum() == 26
assert sorted(judge.loc[judge['規則2'] != judge['人'], 'ID']) == ['D24', 'D26', 'D27', 'D29']
assert judge['人'].value_counts().to_dict() == {'困りごと': 18, '否定': 9, '保留': 3}
print('判定の比較は記録と一致しています。')

判定の比較は記録と一致しています。


**記録（判定）**：日誌×語の30組を人が読むと、困りごと **18**・否定 **9**・保留 **3**。
- 素朴な規則の一致は **18/30（60%）**。「人手が足り**ない**」（D04・D13・D36 など）を否定と誤る。
- 規則2の一致は **26/30（87%）**。残る4組は**規則では決められない**種類：D24「故障**かと思ったが**種詰まり」（推測の打ち消し）、D26「**苗の生育**に遅れ」（対象が作業でない）、D27「足りない**かもしれない**」（予測）、D29「故障していた草刈機が**修理から戻った**」（解消の報告）。
- 規則2は**この36件を見てから作った後付け**なので、新しい日誌で同じ一致率が出るとは限らない。


### 検索の漏れ：3語では拾えない言い換え
規則の精度は「拾ったもの」についてしか測れません。**拾えなかった困りごと**は、全件を読むか、言い換えの辞書で探します。
`(?<!人)` は「直前が『人』でない」という条件で、「人手が足りない」を二重に数えないための工夫です。


In [11]:
SYNONYMS = {'遅れ': [r'遅延'], '故障': [r'壊れ', r'動かなくなっ'], '人手': [r'(?<!人)手が足り']}
base = df['本文'].str.contains('|'.join(KEYWORDS))
extra = []
for kw, pats in SYNONYMS.items():
    for p in pats:
        for _, r in df[df['本文'].str.contains(p, regex=True) & ~base].iterrows():
            extra.append({'ID': r['ID'], '語': kw, '作業': r['作業'], '月': r['月'], '本文': r['本文'],
                          '規則2': rule2(r['本文'], kw)})
extra = pd.DataFrame(extra)
assert extra['ID'].tolist() == ['D32', 'D23', 'D15', 'D07']
extra

,ID,語,作業,月,本文,規則2
0,D32,遅れ,育苗,2026-10,遅延なし。,否定
1,D23,故障,育苗,2026-09,育苗ハウスの換気扇が壊れた。,困りごと
2,D15,故障,機械整備,2026-08,かん水ポンプが動かなくなった。,困りごと
3,D07,人手,収穫,2026-07,収穫量が多く、手が足りない。,困りごと


→ 言い換えで**4件**増え、そのうち3件（D07・D15・D23）は困りごと、D32「遅延なし」は否定。3語だけの検索では、困りごとを**少なめに**見積もる。

## ②｜数値を可視化する：どの作業・どの月に困りごとが書かれているか
人が**原文で確かめた**困りごとだけを使います（保留・否定は除く）。言い換えで見つけた分を含めるかは、下の1行で切り替えます。


In [12]:
INCLUDE_SYNONYMS = True   # ← 言い換えで見つけた困りごとも含めるか（1か所だけ変えて比べる）

confirmed = judge[judge['人'] == '困りごと'].merge(df[['ID', '作業', '月']], on='ID')[['ID', '語', '作業', '月']]
if INCLUDE_SYNONYMS:
    confirmed = pd.concat([confirmed, extra[extra['規則2'] == '困りごと'][['ID', '語', '作業', '月']]])

# 単位は「日誌×語」。D04 のように1件の日誌に『遅れ』と『人手』があれば2と数える
pivot = confirmed.pivot_table(index='作業', columns='月', values='ID', aggfunc='count', fill_value=0)
pivot.loc['合計'] = pivot.sum()
pivot['合計'] = pivot.sum(axis=1)
pivot

月,2026-07,2026-08,2026-09,2026-10,合計
作業,,,,,
出荷,1,1,2,1,5
収穫,3,1,0,0,4
播種,0,0,1,1,2
機械整備,2,1,1,0,4
育苗,0,0,1,0,1
除草,1,2,0,2,5
合計,7,5,5,4,21


In [13]:
import plotly.express as px

heat = pivot.drop(index='合計', columns='合計')
fig = px.imshow(heat, text_auto=True, aspect='auto', color_continuous_scale='YlOrBr',
                labels=dict(x='月', y='作業', color='困りごと（日誌×語）'),
                title='原文で確かめた困りごと：作業 × 月')
fig.update_layout(height=380, margin=dict(l=10, r=10, t=50, b=10))
fig.show()

by_kw = confirmed.groupby(['語', '作業']).size().reset_index(name='件数')
fig2 = px.bar(by_kw, x='件数', y='語', color='作業', orientation='h',
              title='困りごとの種類 × 作業（何についての困りごとか）')
fig2.update_layout(height=300, margin=dict(l=10, r=10, t=50, b=10))
fig2.show()

In [14]:
# 自己チェック（INCLUDE_SYNONYMS = True のとき）
if INCLUDE_SYNONYMS:
    assert len(confirmed) == 21
    assert confirmed['語'].value_counts().to_dict() == {'遅れ': 10, '故障': 5, '人手': 6}
    assert pivot.loc['合計', '合計'] == 21
    print('②の数字は記録と一致しています。')

②の数字は記録と一致しています。


**記録（②）**：困りごとは日誌×語で **21**（3語で18＋言い換えで3）。内訳は遅れ10・故障5・人手6。
「遅れ」は除草・出荷・播種に散らばり、「故障」は**機械整備の日誌に集中**、「人手」は収穫・除草・出荷に出て、6件中4件が**7〜8月**。
ただし**作業の日数自体が違う**（機械整備の日は7日、育苗は3日）ので、セルの数字をそのまま「その作業が一番大変」とは読めない。下のセルで日数で割ると、出荷・除草・収穫が0.7前後で並び、**作業間の差は小さい**。


In [15]:
# 作業ごとの日誌の日数（分母）で割って比べる
days = df.groupby('作業').size().rename('日誌の日数')
rate = pd.concat([days, confirmed.groupby('作業').size().rename('困りごと')], axis=1).fillna(0)
rate['日誌1件あたり'] = (rate['困りごと'] / rate['日誌の日数']).round(2)
rate.sort_values('日誌1件あたり', ascending=False)

,日誌の日数,困りごと,日誌1件あたり
作業,,,
出荷,7,5,0.71
除草,7,5,0.71
収穫,6,4,0.67
機械整備,7,4,0.57
播種,6,2,0.33
育苗,3,1,0.33


## ③｜自分の条件で判断する：AIエージェントを最初に入れる作業はどれか
ここからは**答えが人によって違います**。下の候補は、困りごとの種類から考えた**仮置きの対応づけ**です。
自分の法人の条件（使えるデータ、導入にかけられる時間、困りごとで失っている時間）を入れて、最初の1つを決めます。


In [16]:
CANDIDATES = pd.DataFrame([
    # 困りごと, AIエージェントの候補,                         前提として必要なデータ
    ('遅れ', '天気予報と作業予定を突き合わせ、段取りの組み直し案を出す', '作業予定表・気象予報'),
    ('故障', '機械ごとの故障履歴を検索し、点検・部品交換を知らせる',       '機械台帳・故障の記録'),
    ('人手', '作業量と出勤予定から、人手が足りない日を前もって知らせる',     '出勤予定・作業ごとの所要時間'),
], columns=['語', 'エージェント候補', '必要なデータ'])

# ▼ 自分の条件（ここを書きかえる）
MY_CONDITIONS = {
    # 困りごと1件あたり、おおよそ何時間失っているか（日誌には書かれていない → 現場で聞いて入れる）
    '1件あたりの損失時間(h)': {'遅れ': 2.0, '故障': 4.0, '人手': 3.0},
    # 必要なデータがもう手元にあるか（あれば1、なければ0.5 などで割り引く）
    'データの準備度': {'遅れ': 1.0, '故障': 0.5, '人手': 0.5},
}

decide = CANDIDATES.copy()
decide['確認済みの件数'] = decide['語'].map(confirmed['語'].value_counts()).fillna(0).astype(int)
decide['損失時間の見込み(h)'] = decide['確認済みの件数'] * decide['語'].map(MY_CONDITIONS['1件あたりの損失時間(h)'])
decide['優先度'] = (decide['損失時間の見込み(h)'] * decide['語'].map(MY_CONDITIONS['データの準備度'])).round(1)
decide.sort_values('優先度', ascending=False)

,語,エージェント候補,必要なデータ,確認済みの件数,損失時間の見込み(h),優先度
0,遅れ,天気予報と作業予定を突き合わせ、段取りの組み直し案を出す,作業予定表・気象予報,10,20.0,20.0
1,故障,機械ごとの故障履歴を検索し、点検・部品交換を知らせる,機械台帳・故障の記録,5,20.0,10.0
2,人手,作業量と出勤予定から、人手が足りない日を前もって知らせる,出勤予定・作業ごとの所要時間,6,18.0,9.0


**判断メモ（記入例）**：上の仮の条件では「遅れ」の段取り組み直しが最上位になる（件数が多く、作業予定表と気象予報はすぐ用意できるため）。
「故障」は件数は少ないが1件の損失が大きいので、**機械台帳が整えば「遅れ」と並ぶ（どちらも20.0）**。`データの準備度` の故障を 1.0 に変えて確かめること。

**この判断に必要なのに、日誌に書かれていないもの**：困りごとで**止まった時間**、**影響した量**（出荷数など）、**誰が対応したか**。
→ 次の一手は分析ではなく**日誌の書式の改訂**：自由記述は残したまま、「止まった時間（分）」「影響（出荷・作業・品質）」の2欄を足す。


## まとめ（3行）
- **観測**：検索規則＝部分文字列の一致。分母は匿名化済みの日誌36件。「遅れ」14回・14件、「故障」8回・7件、「人手」9回・9件。
- **解釈と根拠**：一致した30組を読むと、困りごとは18組（言い換えを含めると21）。否定9組は「遅れはない」「故障なし」「人手は足りた」の形で、**語の直後**で判定できた（規則2の一致 26/30）。一方「人手が足り**ない**」は「ない」を含むのに困りごとで、否定の規則は**語ごとに逆向き**になる。「故障」は機械整備の日誌に、「人手」は7〜8月に偏るが、作業ごとの日数で割ると作業間の差は小さい。
- **限界・次の問い**：
  1. **判断保留**：推測の打ち消し（D24）、対象が作業でない（D26）、予測（D27）、解消の報告（D29）は規則では決められない。
  2. **判定者が1人**：2人が独立に読み、一致率（κ係数）を確かめる。
  3. **検索の漏れ**：言い換えで4件見つかった。辞書を広げても、書かれなかった困りごとは数えられない。
  4. **件数≠重要度**：日誌には損失時間が書かれていない。③の優先度は仮の条件に強く依存する。
  5. **後付けの規則**：規則2と言い換え辞書はこの36件を見て作った。**別の月・別の法人の日誌**で試して、一致率が保たれるかを確かめたい。


## 研究メモ（課題3・改訂版）
元のメモから変えたのは4点：①冒頭を「関心分野は〜」にして4項目を明示、②テキストの**単位（1日1件）**を書いて分母を決めた、③数え方を「作業別・月別」にして問い（どの作業や時期に）と対応させた、④このノートで見つかった**言い換えの漏れ**を注意点に加えた。


In [17]:
memo = (
    '関心分野は、小規模農業法人へのAIエージェントの導入である。'
    'テキストは、従業員が毎日記入する作業日誌の自由記述（1日1件）とする。'
    '問いは「作業上の困りごとが、どの作業や時期に、どのような言葉で書かれているか」である。'
    'まず「遅れ」「故障」「人手」を含む日誌を作業別・月別に数え、該当した原文を読む。'
    '注意点は四つある。第一に、語の出現回数と日誌の件数を区別し、分母を示す。'
    '第二に、「故障はなかった」のような否定も一致するため、原文で確かめる。'
    '第三に、「壊れた」などの言い換えは漏れる。'
    '第四に、氏名や取引先名を匿名化してから分析し、外部のAIには入力しない。'
)
print(memo)
print()
print('字数：', len(memo))
for k in ['関心分野', 'テキスト', '問い', '注意点']:
    print(f"  {k}：{'含む' if k in memo else '★不足'}")
assert 200 <= len(memo) <= 300, '200〜300字に収まっていません'

関心分野は、小規模農業法人へのAIエージェントの導入である。テキストは、従業員が毎日記入する作業日誌の自由記述（1日1件）とする。問いは「作業上の困りごとが、どの作業や時期に、どのような言葉で書かれているか」である。まず「遅れ」「故障」「人手」を含む日誌を作業別・月別に数え、該当した原文を読む。注意点は四つある。第一に、語の出現回数と日誌の件数を区別し、分母を示す。第二に、「故障はなかった」のような否定も一致するため、原文で確かめる。第三に、「壊れた」などの言い換えは漏れる。第四に、氏名や取引先名を匿名化してから分析し、外部のAIには入力しない。

字数： 276
  関心分野：含む
  テキスト：含む
  問い：含む
  注意点：含む


## AI利用の記録
| 項目 | 記入 |
|---|---|
| 使ったAI | （例：Claude）※使わなかった場合は「なし」 |
| 用途 | ノートの構成整理、`search()`・KWIC・規則2のコード案 |
| AIに渡したもの | 研究メモと**架空の**日誌のみ（実在の日誌・氏名・取引先名は渡していない） |
| 自分で確認した点 | ① 全セルを上から実行し、自己チェックが通ることを確認 ② `HUMAN` の判定は30組の原文を自分で読んで決めた ③ 研究メモの字数を `len()` で確認 |
